# Public Safety End-to-End Demo

This notebook shows the **system-level flow** of the project:

1. Smart CCTV or a citizen report detects an early public-safety signal
2. AI summarizes the signal and produces a dispatch recommendation
3. A human operator makes the final launch decision
4. The approved event is converted into a drone mission request

It is intentionally lightweight and focuses on orchestration rather than
detector training or drone-control internals.


## System layers

- **Perception layer**: lightweight CCTV detector and report intake
- **Decision-support layer**: temporal verification and dispatch recommendation
- **Human-in-the-loop layer**: final operator approval
- **Drone response layer**: route planning, anomaly detection, and emergency mitigation


In [ ]:
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
import pandas as pd


In [ ]:
@dataclass
class EventSnapshot:
    source: str
    location: str
    detected_objects: List[str]
    external_report: bool
    hotspot: bool


@dataclass
class DispatchDecision:
    level: str
    score: float
    recommended: bool
    reasons: List[str]


@dataclass
class DroneMissionRequest:
    mission_id: str
    event_type: str
    launch_approved: bool
    launch_site: str
    target_location: str
    recommended_mode: str


In [ ]:
def recommend_dispatch(snapshot: EventSnapshot) -> DispatchDecision:
    score = 0.0
    reasons = []

    if "fire" in snapshot.detected_objects:
        score += 0.55
        reasons.append("fire signal detected")
    if "knife" in snapshot.detected_objects:
        score += 0.45
        reasons.append("weapon signal detected")
    if snapshot.external_report:
        score += 0.15
        reasons.append("external report corroborates the event")
    if snapshot.hotspot:
        score += 0.10
        reasons.append("operator-marked hotspot")

    score = min(score, 1.0)

    if score >= 0.85:
        level = "urgent_dispatch_recommended"
    elif score >= 0.60:
        level = "dispatch_recommended"
    elif score >= 0.30:
        level = "operator_review"
    else:
        level = "observe_only"

    return DispatchDecision(
        level=level,
        score=score,
        recommended=level in {"dispatch_recommended", "urgent_dispatch_recommended"},
        reasons=reasons if reasons else ["no meaningful risk signal"],
    )


def build_mission_request(
    snapshot: EventSnapshot,
    decision: DispatchDecision,
    human_approved: bool,
    launch_site: str = "central-rooftop-station",
) -> Optional[DroneMissionRequest]:
    if not human_approved:
        return None

    if "fire" in snapshot.detected_objects:
        event_type = "fire_response"
        mode = "rapid_visual_assessment"
    elif "knife" in snapshot.detected_objects:
        event_type = "security_response"
        mode = "standoff_surveillance"
    else:
        event_type = "general_incident"
        mode = "visual_confirmation"

    return DroneMissionRequest(
        mission_id="demo-mission-001",
        event_type=event_type,
        launch_approved=True,
        launch_site=launch_site,
        target_location=snapshot.location,
        recommended_mode=mode,
    )


In [ ]:
scenarios = [
    EventSnapshot(
        source="smart_cctv",
        location="Sejong block A",
        detected_objects=["fire"],
        external_report=False,
        hotspot=False,
    ),
    EventSnapshot(
        source="smart_cctv + report",
        location="Sejong block B",
        detected_objects=["knife"],
        external_report=True,
        hotspot=True,
    ),
]

rows = []
for i, snapshot in enumerate(scenarios, start=1):
    decision = recommend_dispatch(snapshot)
    human_approved = decision.recommended
    mission = build_mission_request(snapshot, decision, human_approved)

    rows.append(
        {
            "scenario": i,
            "source": snapshot.source,
            "location": snapshot.location,
            "detected_objects": ", ".join(snapshot.detected_objects),
            "dispatch_level": decision.level,
            "dispatch_score": round(decision.score, 2),
            "recommended": decision.recommended,
            "human_approved": human_approved,
            "mission_mode": mission.recommended_mode if mission else "not launched",
        }
    )

    print(f"=== Scenario {i} ===")
    print("snapshot :", asdict(snapshot))
    print("decision :", asdict(decision))
    print("mission  :", asdict(mission) if mission else "launch not approved")
    print()

demo_df = pd.DataFrame(rows)
display(demo_df)


## How this connects to the rest of the repository

- `cctv/` provides the lightweight object detector and dispatch recommendation logic
- `drone/` provides the route planner, anomaly detector, and emergency sequencer
- `integration/` turns those outputs into a single **public-safety first-response** story
